# Разведочный анализ KuaiRand

KuaiRand - датасет логов рекомендаций Kuaishou. Он содержит трафик стандартной рекомендательной политики, трафик случайного показа, последовательные истории взаимодействий, пользовательские признаки, признаки видео/items и несколько feedback-сигналов. Релиз доступен в трех версиях: KuaiRand-Pure, KuaiRand-1K и KuaiRand-27K.

Этот ноутбук написан как исследовательский EDA-документ для работы по рекомендательным системам. Он явно разделяет числа из документации и числа, вычисленные по файлам. Ноутбук не скачивает данные, не создает синтетическую замену и не придумывает отсутствующие результаты.

## Исследовательские вопросы

1. Чем отличаются Pure / 1K / 27K?
2. Как устроены логи взаимодействий?
3. Чем отличаются стандартная рекомендация и случайный показ?
4. Насколько длинные пользовательские истории?
5. Насколько разрежены наблюдаемые взаимодействия?
6. Какие целевые feedback-сигналы доступны?
7. Как устроена временная структура?
8. Какие признаки могут создавать leakage?
9. Какие семейства бенчмарк-протоколов возможны?
10. Какую версию использовать для разработки и полных экспериментов?

## Конфигурация

Ноутбук рассчитан на запуск на cHARISMa после `git pull`. По умолчанию `DATA_ROOT` указывает на серверный путь `/home/daryumin/iberdov/Corpora`. Другой путь можно задать через переменную окружения `KUAIRAND_DATA_ROOT`, но ноутбук не использует автоматический локальный fallback и не скачивает датасет.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "eda_utils.py").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_utils import (  # noqa: E402
    FEEDBACK_COLUMNS,
    INTERACTION_COLUMNS,
    WATCH_TIME_COLUMNS,
    available_columns,
    collect_lazy,
    concat_lazy_frames,
    dataset_inventory,
    discover_kuairand_files,
    human_size,
    lazy_schema_names,
    numeric_summary,
    require_polars,
    safe_percentiles,
    scan_table,
    total_size_bytes,
    version_roots,
)

SERVER_DATA_ROOT = Path("/home/daryumin/iberdov/Corpora")
DATA_ROOT = Path(os.environ.get("KUAIRAND_DATA_ROOT", SERVER_DATA_ROOT)).expanduser()
PURE_ROOT = DATA_ROOT / "KuaiRand-Pure" / "KuaiRand-Pure"
K1_ROOT = DATA_ROOT / "KuaiRand-1K" / "KuaiRand-1K"
K27_ROOT = DATA_ROOT / "KuaiRand-27K" / "KuaiRand-27K"
VERSION_ROOTS = version_roots(DATA_ROOT)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_AVAILABLE = DATA_ROOT.exists()
VERSION_AVAILABLE = {version: root.exists() for version, root in VERSION_ROOTS.items()}

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")
if not DATA_AVAILABLE:
    print(f"KuaiRand data directory not found: {DATA_ROOT}")
else:
    display(pd.DataFrame(
        [
            {"version": version, "path": str(root), "exists": exists}
            for version, root in VERSION_ROOTS.items()
            for exists in [root.exists()]
        ]
    ))


## Версии датасета

Таблица ниже содержит **статистику из документации / README**. Эти числа скопированы из документации датасета и не должны смешиваться с результатами, вычисленными ноутбуком по локальным файлам.

In [ ]:
documentation_stats = pd.DataFrame([
    {
        "version": "KuaiRand-27K",
        "users": 27_285,
        "standard_items": 32_038_725,
        "standard_interactions": 322_278_385,
        "random_items": 7_583,
        "random_interactions": 1_186_059,
        "source": "Статистика из документации / README",
    },
    {
        "version": "KuaiRand-1K",
        "users": 1_000,
        "standard_items": 4_369_953,
        "standard_interactions": 11_713_045,
        "random_items": 7_388,
        "random_interactions": 43_028,
        "source": "Статистика из документации / README",
    },
    {
        "version": "KuaiRand-Pure",
        "users": 27_285,
        "standard_items": "about 7.5K candidate-pool items",
        "standard_interactions": 1_436_609,
        "random_items": 7_583,
        "random_interactions": 1_186_059,
        "source": "Статистика из документации / README",
    },
])
display(documentation_stats)


## Инвентаризация датасета

Этот раздел сканирует только метаданные файловой системы: список файлов, размеры, суффиксы и EDA-категорию для Pure / 1K / 27K, если соответствующие каталоги доступны.

In [ ]:
inventory_rows = []
for version, root in VERSION_ROOTS.items():
    rows = dataset_inventory(root) if root.exists() else []
    for row in rows:
        inventory_rows.append({"version": version, **row})

inventory_df = pd.DataFrame(inventory_rows)
if inventory_df.empty:
    print("Skipped inventory: KuaiRand data directories are not available in DATA_ROOT.")
else:
    display(inventory_df.sort_values(["version", "category", "path"]))

    inventory_summary = (
        inventory_df.groupby(["version", "category"], as_index=False)
        .agg(files=("path", "count"), size_bytes=("size_bytes", "sum"))
        .assign(size=lambda frame: frame["size_bytes"].map(human_size))
        .sort_values(["version", "category"])
    )
    display(inventory_summary)


In [ ]:
disk_summary_rows = []
for version, root in VERSION_ROOTS.items():
    rows = dataset_inventory(root) if root.exists() else []
    disk_summary_rows.append({
        "version": version,
        "path": str(root),
        "exists": root.exists(),
        "files": len(rows),
        "disk_size_bytes": total_size_bytes(rows),
        "disk_size": human_size(total_size_bytes(rows)) if rows else None,
        "source": "Вычислено по локальным файлам" if rows else "Не вычислено: каталог данных отсутствует",
    })

disk_summary = pd.DataFrame(disk_summary_rows)
display(disk_summary)


## Поиск файлов

Код ниже находит ожидаемые таблицы KuaiRand по токенам в имени файла, а не через жестко заданные суффиксы конкретных версий. Это делает ноутбук устойчивее к вариантам имен файлов в Pure / 1K / 27K.

In [ ]:
DISCOVERED_FILES = {
    version: discover_kuairand_files(root) if root.exists() else {}
    for version, root in VERSION_ROOTS.items()
}

discovered_rows = []
for version, files in DISCOVERED_FILES.items():
    for logical_name, path in files.items():
        discovered_rows.append({
            "version": version,
            "logical_name": logical_name,
            "path": str(path) if path is not None else None,
            "exists": path is not None,
        })

discovered_files_df = pd.DataFrame(discovered_rows)
if discovered_files_df.empty:
    print("Skipped file discovery: no version directory is available.")
else:
    display(discovered_files_df.sort_values(["version", "logical_name"]))


## Стандартная рекомендация и случайный показ

KuaiRand разделяет три периода логов взаимодействий:

- `log_standard_4_08_to_4_21`
- `log_standard_4_22_to_5_08`
- `log_random_4_22_to_5_08`

Эти имена описывают policy сбора данных и календарные окна. Их нельзя автоматически переименовывать в train / validation / test. Протокол бенчмарка должен отдельно определить предобработку, split, candidate set и оценивание.

Standard logs собраны под платформенной рекомендательной policy. Random log содержит randomized exposure из candidate pool. Важные поля: `is_rand`, `tab` и документированные 15 сценариев рекомендаций.

## Зачем нужен random exposure

Random exposure важен, потому что рекомендательные логи подвержены exposure bias и selection bias: наблюдаемый feedback условен на том, что предыдущая policy решила показать пользователю. Рандомизированный показ может быть полезен для debiasing, off-policy evaluation и causal recommendation. В этом EDA мы описываем ассоциации и различия распределений; causal effect без отдельного causal design не утверждается.

## Схема логов взаимодействий

Таблица ниже описывает ожидаемые колонки логов взаимодействий, их смысл и возможную роль в моделировании. Фактические dtypes проверяются по локальным файлам в следующих разделах.

In [ ]:
interaction_schema = pd.DataFrame([
    {"column": "user_id", "смысл": "Анонимизированный идентификатор пользователя", "роль в моделировании": "User key, ключ группировки для sequences"},
    {"column": "video_id", "смысл": "Анонимизированный идентификатор видео/item", "роль в моделировании": "Item key, target item для ranking"},
    {"column": "date", "смысл": "Календарная дата показа/взаимодействия", "роль в моделировании": "Temporal split, анализ трендов"},
    {"column": "hourmin", "смысл": "Время взаимодействия в формате hour-minute", "роль в моделировании": "Time-of-day feature после валидации"},
    {"column": "time_ms", "смысл": "Timestamp в миллисекундах", "роль в моделировании": "Сортировка sequence, chronological split"},
    {"column": "is_click", "смысл": "Click feedback", "роль в моделировании": "Binary target или auxiliary feedback"},
    {"column": "is_like", "смысл": "Like feedback", "роль в моделировании": "Sparse positive target или auxiliary task"},
    {"column": "is_follow", "смысл": "Follow feedback", "роль в моделировании": "Sparse engagement target"},
    {"column": "is_comment", "смысл": "Comment feedback", "роль в моделировании": "Sparse engagement target"},
    {"column": "is_forward", "смысл": "Forward/share feedback", "роль в моделировании": "Sparse engagement target"},
    {"column": "is_hate", "смысл": "Negative feedback", "роль в моделировании": "Negative preference / safety signal"},
    {"column": "long_view", "смысл": "Индикатор long-view", "роль в моделировании": "Binary watch-time target"},
    {"column": "play_time_ms", "смысл": "Наблюдаемое время просмотра", "роль в моделировании": "Continuous engagement signal"},
    {"column": "duration_ms", "смысл": "Длительность видео", "роль в моделировании": "Делитель для нормализации; item-side feature с осторожностью"},
    {"column": "profile_stay_time", "смысл": "Время на профиле", "роль в моделировании": "Auxiliary engagement signal"},
    {"column": "comment_stay_time", "смысл": "Время в комментариях", "роль в моделировании": "Auxiliary engagement signal"},
    {"column": "is_profile_enter", "смысл": "Индикатор перехода в профиль", "роль в моделировании": "Binary auxiliary feedback"},
    {"column": "is_rand", "смысл": "Флаг random exposure", "роль в моделировании": "Policy indicator; нельзя смешивать без контроля"},
    {"column": "tab", "смысл": "Recommendation scenario / tab", "роль в моделировании": "Scenario/context feature или stratification key"},
])
interaction_schema["dtype"] = "проверяется по файлам ниже"
display(interaction_schema[["column", "dtype", "смысл", "роль в моделировании"]])


## Вспомогательные функции ноутбука

Функции ниже сохраняют ленивый доступ к данным и защищают разделы, зависящие от наличия датасета. Pure и 1K можно анализировать интерактивно после агрегирования. Полный 27K должен обрабатываться через `src/eda_27k.py`, а не через полный scan в ноутбуке.

In [ ]:
LOG_META = {
    "standard_early": {"policy": "standard", "label": "log_standard_4_08_to_4_21"},
    "standard_late": {"policy": "standard", "label": "log_standard_4_22_to_5_08"},
    "random": {"policy": "random", "label": "log_random_4_22_to_5_08"},
}

MAX_PLOT_ROWS = 200_000


def version_ready(version: str) -> bool:
    return bool(VERSION_ROOTS.get(version, Path("missing")).exists())


def skip(reason: str) -> None:
    print(f"Skipped: {reason}")


def pl_df_to_pandas(frame):
    return pd.DataFrame(frame.to_dicts())


def schema_records(lf) -> list[dict[str, str]]:
    if hasattr(lf, "collect_schema"):
        schema = lf.collect_schema()
        return [{"column": name, "dtype": str(schema[name])} for name in schema.names()]
    return [{"column": name, "dtype": str(dtype)} for name, dtype in lf.schema.items()]


def scan_interaction_log(version: str, log_key: str):
    if not version_ready(version):
        return None
    path = DISCOVERED_FILES.get(version, {}).get(log_key)
    if path is None:
        return None
    pl = require_polars()
    meta = LOG_META[log_key]
    return scan_table(path).with_columns([
        pl.lit(meta["policy"]).alias("policy"),
        pl.lit(meta["label"]).alias("source_log"),
    ])


def combine_interactions(version: str, policies: tuple[str, ...] = ("standard", "random")):
    if not version_ready(version):
        return None
    scans = []
    for log_key, meta in LOG_META.items():
        if meta["policy"] not in policies:
            continue
        lf = scan_interaction_log(version, log_key)
        if lf is not None:
            scans.append(lf)
    if not scans:
        return None
    return concat_lazy_frames(scans)


def table_or_message(df: pd.DataFrame, message: str) -> None:
    if df.empty:
        print(message)
    else:
        display(df)


## Фактические dtypes логов взаимодействий

Этот раздел проверяет dtypes из схем файлов и не материализует полный датасет.

In [ ]:
schema_rows = []
for version in ("Pure", "1K"):
    if not version_ready(version):
        continue
    for log_key, meta in LOG_META.items():
        path = DISCOVERED_FILES.get(version, {}).get(log_key)
        if path is None:
            continue
        lf = scan_table(path)
        for row in schema_records(lf):
            schema_rows.append({
                "version": version,
                "log": meta["label"],
                **row,
            })

schema_df = pd.DataFrame(schema_rows)
table_or_message(schema_df, "Нет доступных схем interaction logs для Pure/1K.")


## Качество данных: Pure и 1K

Для Pure и 1K раздел вычисляет число строк, пользователей, видео, дубликаты строк, пропуски, диапазоны timestamp/date, значения `tab` и `is_rand`. Эти значения помечаются как **вычислено по локальным файлам** только если файлы доступны.

In [ ]:
def interaction_quality(version: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not version_ready(version):
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    pl = require_polars()
    quality_rows = []
    missing_rows = []
    binary_rows = []

    for log_key, meta in LOG_META.items():
        path = DISCOVERED_FILES.get(version, {}).get(log_key)
        if path is None:
            continue
        lf = scan_table(path)
        names = lazy_schema_names(lf)
        name_set = set(names)

        exprs = [pl.len().alias("rows")]
        if names:
            exprs.append(pl.struct(names).is_duplicated().sum().alias("duplicates"))
            exprs.append(
                pl.sum_horizontal([pl.col(col).is_null().cast(pl.Int64) for col in names]).sum().alias("missing_cells")
            )
        if "user_id" in name_set:
            exprs.append(pl.col("user_id").n_unique().alias("users"))
        if "video_id" in name_set:
            exprs.append(pl.col("video_id").n_unique().alias("videos"))
        if "time_ms" in name_set:
            exprs.extend([
                pl.col("time_ms").cast(pl.Int64, strict=False).min().alias("time_ms_min"),
                pl.col("time_ms").cast(pl.Int64, strict=False).max().alias("time_ms_max"),
            ])
        if "date" in name_set:
            exprs.extend([pl.col("date").min().alias("date_min"), pl.col("date").max().alias("date_max")])
        if "tab" in name_set:
            exprs.append(pl.col("tab").n_unique().alias("tab_n_values"))
        if "is_rand" in name_set:
            exprs.append(pl.col("is_rand").n_unique().alias("is_rand_n_values"))

        quality = collect_lazy(lf.select(exprs)).to_dicts()[0]
        quality_rows.append({
            "version": version,
            "log": meta["label"],
            "policy": meta["policy"],
            "source": "Вычислено по локальным файлам",
            **quality,
        })

        missing = collect_lazy(lf.select([pl.col(col).is_null().sum().alias(col) for col in names])).to_dicts()[0]
        for column, missing_count in missing.items():
            if missing_count:
                missing_rows.append({
                    "version": version,
                    "log": meta["label"],
                    "column": column,
                    "missing_count": missing_count,
                    "source": "Вычислено по локальным файлам",
                })

        for column in [col for col in FEEDBACK_COLUMNS if col in name_set]:
            value = pl.col(column).cast(pl.Int64, strict=False)
            binary_quality = collect_lazy(lf.select([
                value.is_null().sum().alias("missing_count"),
                value.n_unique().alias("unique_values"),
                (value.is_not_null() & (~value.is_in([0, 1]))).sum().alias("invalid_count"),
                value.min().alias("min_value"),
                value.max().alias("max_value"),
            ])).to_dicts()[0]
            binary_rows.append({
                "version": version,
                "log": meta["label"],
                "column": column,
                **binary_quality,
                "source": "Вычислено по локальным файлам",
            })

    return pd.DataFrame(quality_rows), pd.DataFrame(missing_rows), pd.DataFrame(binary_rows)


quality_tables = {}
missing_tables = {}
binary_tables = {}
for version in ("Pure", "1K"):
    quality, missing, binary = interaction_quality(version)
    quality_tables[version] = quality
    missing_tables[version] = missing
    binary_tables[version] = binary

quality_df = pd.concat([df for df in quality_tables.values() if not df.empty], ignore_index=True) if any(not df.empty for df in quality_tables.values()) else pd.DataFrame()
missing_df = pd.concat([df for df in missing_tables.values() if not df.empty], ignore_index=True) if any(not df.empty for df in missing_tables.values()) else pd.DataFrame()
binary_quality_df = pd.concat([df for df in binary_tables.values() if not df.empty], ignore_index=True) if any(not df.empty for df in binary_tables.values()) else pd.DataFrame()

table_or_message(quality_df, "Data-quality summary для Pure/1K не вычислен.")
table_or_message(missing_df, "Missing values не найдены или данные недоступны.")
table_or_message(binary_quality_df, "Качество binary feedback summary не вычислен.")


In [ ]:
def distinct_values_for_columns(version: str, columns: tuple[str, ...] = ("tab", "is_rand")) -> pd.DataFrame:
    if not version_ready(version):
        return pd.DataFrame()
    pl = require_polars()
    rows = []
    for log_key, meta in LOG_META.items():
        path = DISCOVERED_FILES.get(version, {}).get(log_key)
        if path is None:
            continue
        lf = scan_table(path)
        names = set(lazy_schema_names(lf))
        for column in columns:
            if column not in names:
                continue
            values = collect_lazy(lf.select(pl.col(column).drop_nulls().unique().sort()).limit(100)).to_dicts()
            rows.append({
                "version": version,
                "log": meta["label"],
                "column": column,
                "values_preview": [row[column] for row in values],
                "source": "Вычислено по локальным файлам",
            })
    return pd.DataFrame(rows)

value_previews = pd.concat(
    [distinct_values_for_columns(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(value_previews, "Превью значений tab/is_rand не вычислено.")


## Количество записей по standard и random

Этот раздел вычисляет `Version`, `Policy`, `Users`, `Items`, `Interactions` для Pure и 1K. 27K намеренно исключен из интерактивного ноутбука, чтобы не запускать тяжелые scans.

In [ ]:
def policy_counts_for_version(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    exprs = [pl.len().alias("Interactions")]
    if "user_id" in names:
        exprs.append(pl.col("user_id").n_unique().alias("Users"))
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("Items"))
    records = collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts()
    df = pd.DataFrame(records).rename(columns={"policy": "Policy"})
    if df.empty:
        return df
    df.insert(0, "Version", version)
    total = df["Interactions"].sum()
    df["interaction_share"] = df["Interactions"] / total if total else np.nan
    df["source"] = "Вычислено по локальным файлам"
    return df

policy_counts_df = pd.concat(
    [policy_counts_for_version(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(policy_counts_df, "Counts standard-vs-random для Pure/1K не вычислены.")


## Feedback-сигналы

Анализируются feedback-сигналы `is_click`, `is_like`, `is_follow`, `is_comment`, `is_forward`, `is_hate`, `long_view`, `is_profile_enter`. Числа positives и rates считаются отдельно для standard и random exposure.

In [ ]:
def feedback_by_policy(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    feedback_cols = available_columns(lf, FEEDBACK_COLUMNS)
    if not feedback_cols:
        return pd.DataFrame()

    exprs = []
    for column in feedback_cols:
        value = pl.col(column).cast(pl.Float64, strict=False)
        exprs.extend([
            value.sum().alias(f"{column}__positive_count"),
            value.mean().alias(f"{column}__positive_rate"),
            pl.col(column).is_null().sum().alias(f"{column}__missing_count"),
        ])
    wide = collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts()

    rows = []
    for row in wide:
        for column in feedback_cols:
            rows.append({
                "version": version,
                "policy": row["policy"],
                "signal": column,
                "positive_count": row.get(f"{column}__positive_count"),
                "positive_rate": row.get(f"{column}__positive_rate"),
                "missing_count": row.get(f"{column}__missing_count"),
                "source": "Вычислено по локальным файлам",
            })
    return pd.DataFrame(rows)

feedback_rates_df = pd.concat(
    [feedback_by_policy(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(feedback_rates_df, "Таблица feedback rates не вычислена.")


In [ ]:
if feedback_rates_df.empty:
    skip("для графика feedback rates нужны данные Pure или 1K")
else:
    plot_version = "1K" if "1K" in set(feedback_rates_df["version"]) else feedback_rates_df["version"].iloc[0]
    pivot = feedback_rates_df[feedback_rates_df["version"] == plot_version].pivot(
        index="signal", columns="policy", values="positive_rate"
    )
    ax = pivot.plot(kind="bar", figsize=(10, 4), rot=35)
    ax.set_title(f"Positive feedback rate по policy: KuaiRand-{plot_version}")
    ax.set_ylabel("positive rate")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()


## Анализ watch-time

`play_ratio = play_time_ms / duration_ms` создается только в памяти. Для строк с `duration_ms <= 0` значение `play_ratio` становится null. Clipping используется только для визуализации и не меняет raw values.

In [ ]:
def with_play_ratio(lf):
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if not {"play_time_ms", "duration_ms"}.issubset(names):
        return lf
    return lf.with_columns(
        pl.when(pl.col("duration_ms").cast(pl.Float64, strict=False) > 0)
        .then(pl.col("play_time_ms").cast(pl.Float64, strict=False) / pl.col("duration_ms").cast(pl.Float64, strict=False))
        .otherwise(None)
        .alias("play_ratio")
    )


def watch_time_percentiles(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    lf = with_play_ratio(lf)
    names = set(lazy_schema_names(lf))
    cols = [col for col in ("play_time_ms", "duration_ms", "play_ratio") if col in names]
    if not cols:
        return pd.DataFrame()

    exprs = []
    for column in cols:
        value = pl.col(column).cast(pl.Float64, strict=False)
        exprs.extend([
            value.min().alias(f"{column}_min"),
            value.quantile(0.25).alias(f"{column}_p25"),
            value.quantile(0.50).alias(f"{column}_p50"),
            value.quantile(0.75).alias(f"{column}_p75"),
            value.quantile(0.90).alias(f"{column}_p90"),
            value.quantile(0.95).alias(f"{column}_p95"),
            value.quantile(0.99).alias(f"{column}_p99"),
            value.max().alias(f"{column}_max"),
        ])
    records = collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts()
    df = pd.DataFrame(records)
    if not df.empty:
        df.insert(0, "version", version)
        df["source"] = "Вычислено по локальным файлам; raw percentiles до clipping для визуализации"
    return df

watch_time_summary_df = pd.concat(
    [watch_time_percentiles(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(watch_time_summary_df, "Watch-time percentiles не вычислены.")


In [ ]:
def watch_time_visual_sample(version: str, limit: int = MAX_PLOT_ROWS) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    lf = with_play_ratio(lf)
    names = set(lazy_schema_names(lf))
    columns = [col for col in ["policy", "play_time_ms", "duration_ms", "play_ratio", "is_click", "long_view"] if col in names]
    if not columns:
        return pd.DataFrame()
    records = collect_lazy(lf.select(columns).drop_nulls(subset=["play_ratio"]).limit(limit)).to_dicts()
    return pd.DataFrame(records)

plot_version = "1K" if version_ready("1K") else "Pure"
watch_sample_df = watch_time_visual_sample(plot_version) if version_ready(plot_version) else pd.DataFrame()
if watch_sample_df.empty:
    skip("для watch-time plots нужны interaction logs Pure или 1K")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    watch_sample_df["duration_ms"].clip(upper=watch_sample_df["duration_ms"].quantile(0.99)).hist(ax=axes[0], bins=60)
    axes[0].set_title("Распределение duration, p99 clipping только для графика")
    axes[0].set_xlabel("duration_ms")

    watch_sample_df["play_time_ms"].clip(upper=watch_sample_df["play_time_ms"].quantile(0.99)).hist(ax=axes[1], bins=60)
    axes[1].set_title("Распределение play time, p99 clipping только для графика")
    axes[1].set_xlabel("play_time_ms")

    watch_sample_df["play_ratio"].clip(upper=watch_sample_df["play_ratio"].quantile(0.99)).hist(ax=axes[2], bins=60)
    axes[2].set_title("Play ratio, p99 clipping только для графика")
    axes[2].set_xlabel("play_ratio")
    plt.tight_layout()
    plt.show()

    for target in [col for col in ("is_click", "long_view") if col in watch_sample_df.columns]:
        ax = watch_sample_df.assign(
            play_ratio_plot=lambda df: df["play_ratio"].clip(upper=df["play_ratio"].quantile(0.99))
        ).boxplot(column="play_ratio_plot", by=target, figsize=(6, 4))
        ax.set_title(f"Play ratio vs {target}; clipping только для визуализации")
        ax.set_xlabel(target)
        ax.set_ylabel("play_ratio")
        plt.suptitle("")
        plt.tight_layout()
        plt.show()


## Сдвиг распределений: standard vs random

Раздел сравнивает standard-policy и random-exposure logs по feedback rates, watch-time summaries, числу unique items и interactions per user. Различия интерпретируются как ассоциации / различия распределений, а не как causal effects.

In [ ]:
def distribution_shift_summary(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    lf = with_play_ratio(lf)
    names = set(lazy_schema_names(lf))

    exprs = [pl.len().alias("interactions")]
    if "user_id" in names:
        exprs.append(pl.col("user_id").n_unique().alias("users"))
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("unique_items"))
    for column in ["is_click", "long_view", "is_like", "is_follow", "is_comment"]:
        if column in names:
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).mean().alias(f"{column}_rate"))
    for column in ["play_time_ms", "duration_ms", "play_ratio"]:
        if column in names:
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).mean().alias(f"{column}_mean"))
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).quantile(0.5).alias(f"{column}_median"))

    base = pd.DataFrame(collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts())
    if base.empty:
        return base

    if "user_id" in names:
        per_user = pd.DataFrame(collect_lazy(
            lf.group_by(["policy", "user_id"]).agg(pl.len().alias("interactions_per_user"))
            .group_by("policy")
            .agg([
                pl.col("interactions_per_user").mean().alias("interactions_per_user_mean"),
                pl.col("interactions_per_user").quantile(0.5).alias("interactions_per_user_median"),
                pl.col("interactions_per_user").quantile(0.9).alias("interactions_per_user_p90"),
            ])
            .sort("policy")
        ).to_dicts())
        base = base.merge(per_user, on="policy", how="left")

    base.insert(0, "version", version)
    base["source"] = "Вычислено по локальным файлам"
    return base

shift_summary_df = pd.concat(
    [distribution_shift_summary(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(shift_summary_df, "Распределение-shift summary не вычислен.")


In [ ]:
if shift_summary_df.empty:
    skip("для distribution-shift plots нужны computed summary tables")
else:
    plot_version = "1K" if "1K" in set(shift_summary_df["version"]) else shift_summary_df["version"].iloc[0]
    plot_df = shift_summary_df[shift_summary_df["version"] == plot_version].set_index("policy")

    rate_cols = [col for col in plot_df.columns if col.endswith("_rate")]
    if rate_cols:
        ax = plot_df[rate_cols].T.plot(kind="bar", figsize=(10, 4), rot=35)
        ax.set_title(f"Различия feedback rates по policy: KuaiRand-{plot_version}")
        ax.set_ylabel("rate")
        plt.tight_layout()
        plt.show()

    mean_cols = [col for col in ["play_time_ms_mean", "duration_ms_mean", "play_ratio_mean"] if col in plot_df.columns]
    if mean_cols:
        ax = plot_df[mean_cols].T.plot(kind="bar", figsize=(8, 4), rot=35)
        ax.set_title(f"Различия watch-time distributions: KuaiRand-{plot_version}")
        ax.set_ylabel("mean value")
        plt.tight_layout()
        plt.show()

    if "interactions_per_user_mean" in plot_df.columns:
        ax = plot_df[["interactions_per_user_mean", "interactions_per_user_median", "interactions_per_user_p90"]].T.plot(
            kind="bar", figsize=(8, 4), rot=35
        )
        ax.set_title(f"Interactions per user по policy: KuaiRand-{plot_version}")
        ax.set_ylabel("interactions")
        plt.tight_layout()
        plt.show()


## Анализ `tab`

`tab` описывает recommendation scenario/context. Перед использованием как feature, filter или evaluation stratum его нужно отдельно изучить.

In [ ]:
def tab_analysis(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if "tab" not in names:
        return pd.DataFrame()
    exprs = [pl.len().alias("interactions")]
    if "user_id" in names:
        exprs.append(pl.col("user_id").n_unique().alias("users"))
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("items"))
    for column in ["is_click", "long_view", "is_like"]:
        if column in names:
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).mean().alias(f"{column}_rate"))

    df = pd.DataFrame(collect_lazy(lf.group_by("tab").agg(exprs).sort("interactions", descending=True)).to_dicts())
    if df.empty:
        return df
    df.insert(0, "version", version)
    df["share"] = df["interactions"] / df["interactions"].sum()
    df["source"] = "Вычислено по локальным файлам"
    return df

tab_df = tab_analysis("1K") if version_ready("1K") else tab_analysis("Pure")
table_or_message(tab_df, "Tab analysis не вычислен.")


In [ ]:
if tab_df.empty:
    skip("для tab distribution plot нужна computed tab table")
else:
    ax = tab_df.sort_values("interactions", ascending=False).plot(
        x="tab", y="interactions", kind="bar", figsize=(10, 4), legend=False, rot=35
    )
    ax.set_title("Распределение interactions по tab")
    ax.set_xlabel("tab")
    ax.set_ylabel("interactions")
    plt.tight_layout()
    plt.show()


## Пользовательские признаки

Ожидаемые user features включают степень активности, периоды низкой активности, флаги creator/live-streamer, счетчики и диапазоны follow/fans/friend, возраст регистрации и зашифрованные категориальные колонки `onehot_feat*`.

`onehot_feat*` следует трактовать как зашифрованные категориальные признаки. Их labels напрямую не интерпретируются, но cardinality и missingness важны для моделирования.

In [ ]:
USER_FEATURE_COLUMNS = (
    "user_active_degree",
    "is_lowactive_period",
    "is_live_streamer",
    "is_video_author",
    "follow_user_num",
    "follow_user_num_range",
    "fans_user_num",
    "fans_user_num_range",
    "friend_user_num",
    "friend_user_num_range",
    "register_days",
    "register_days_range",
)


def feature_table_path(version: str, key: str) -> Path | None:
    if not version_ready(version):
        return None
    return DISCOVERED_FILES.get(version, {}).get(key)


def table_shape_and_schema(path: Path) -> tuple[dict[str, int], pd.DataFrame]:
    pl = require_polars()
    lf = scan_table(path)
    row_count = collect_lazy(lf.select(pl.len().alias("rows"))).to_dicts()[0]["rows"]
    schema = pd.DataFrame(schema_records(lf))
    return {"rows": row_count, "columns": len(schema)}, schema


def missing_and_cardinality(path: Path, columns: list[str]) -> pd.DataFrame:
    pl = require_polars()
    lf = scan_table(path)
    names = set(lazy_schema_names(lf))
    selected = [column for column in columns if column in names]
    if not selected:
        return pd.DataFrame()
    exprs = []
    for column in selected:
        exprs.extend([
            pl.col(column).is_null().sum().alias(f"{column}__missing"),
            pl.col(column).n_unique().alias(f"{column}__cardinality"),
        ])
    row = collect_lazy(lf.select(exprs)).to_dicts()[0]
    records = []
    for column in selected:
        records.append({
            "column": column,
            "missing": row.get(f"{column}__missing"),
            "cardinality": row.get(f"{column}__cardinality"),
            "source": "Вычислено по локальным файлам",
        })
    return pd.DataFrame(records)


def top_values(path: Path, column: str, limit: int = 20) -> pd.DataFrame:
    pl = require_polars()
    lf = scan_table(path)
    if column not in set(lazy_schema_names(lf)):
        return pd.DataFrame()
    records = collect_lazy(
        lf.group_by(column).agg(pl.len().alias("rows")).sort("rows", descending=True).limit(limit)
    ).to_dicts()
    return pd.DataFrame(records)


feature_version = "1K" if version_ready("1K") else "Pure"
user_path = feature_table_path(feature_version, "user_features")
if user_path is None:
    skip("таблица user features недоступна")
else:
    shape, user_schema_df = table_shape_and_schema(user_path)
    display(pd.DataFrame([{**shape, "version": feature_version, "path": str(user_path), "source": "Вычислено по локальным файлам"}]))
    display(user_schema_df)

    onehot_columns = [row["column"] for row in user_schema_df.to_dict("records") if row["column"].startswith("onehot_feat")]
    user_missing_cardinality = missing_and_cardinality(user_path, list(USER_FEATURE_COLUMNS) + onehot_columns)
    display(user_missing_cardinality)

    for column in ["user_active_degree", "follow_user_num_range", "fans_user_num_range", "register_days_range"]:
        tv = top_values(user_path, column)
        if not tv.empty:
            display(Markdown(f"### Топ значений: `{column}`"))
            display(tv)


## Базовые признаки видео

Video basic features описывают item metadata: author, type, upload information, visibility, dimensions, duration, music и tags. Это item-side features, но их availability и timestamp semantics нужно проверить до моделирования.

In [ ]:
VIDEO_BASIC_COLUMNS = (
    "author_id",
    "video_type",
    "upload_dt",
    "upload_type",
    "visible_status",
    "video_duration",
    "server_width",
    "server_height",
    "music_id",
    "music_type",
    "tag",
)


def numeric_column_summary_from_scan(path: Path, columns: list[str]) -> pd.DataFrame:
    pl = require_polars()
    lf = scan_table(path)
    names = set(lazy_schema_names(lf))
    selected = [column for column in columns if column in names]
    if not selected:
        return pd.DataFrame()
    exprs = []
    for column in selected:
        value = pl.col(column).cast(pl.Float64, strict=False)
        exprs.extend([
            value.count().alias(f"{column}__count"),
            value.min().alias(f"{column}__min"),
            value.mean().alias(f"{column}__mean"),
            value.quantile(0.5).alias(f"{column}__median"),
            value.quantile(0.9).alias(f"{column}__p90"),
            value.quantile(0.99).alias(f"{column}__p99"),
            value.max().alias(f"{column}__max"),
        ])
    row = collect_lazy(lf.select(exprs)).to_dicts()[0]
    records = []
    for column in selected:
        records.append({
            "column": column,
            "count": row.get(f"{column}__count"),
            "min": row.get(f"{column}__min"),
            "mean": row.get(f"{column}__mean"),
            "median": row.get(f"{column}__median"),
            "p90": row.get(f"{column}__p90"),
            "p99": row.get(f"{column}__p99"),
            "max": row.get(f"{column}__max"),
            "source": "Вычислено по локальным файлам",
        })
    return pd.DataFrame(records)


video_basic_path = feature_table_path(feature_version, "video_basic")
if video_basic_path is None:
    skip("таблица video basic features недоступна")
else:
    shape, video_basic_schema_df = table_shape_and_schema(video_basic_path)
    display(pd.DataFrame([{**shape, "version": feature_version, "path": str(video_basic_path), "source": "Вычислено по локальным файлам"}]))
    display(video_basic_schema_df)

    for column in ["video_type", "upload_type", "visible_status", "music_type"]:
        tv = top_values(video_basic_path, column)
        if not tv.empty:
            display(Markdown(f"### Распределение: `{column}`"))
            display(tv)

    duration_summary = numeric_column_summary_from_scan(video_basic_path, ["video_duration", "server_width", "server_height"])
    table_or_message(duration_summary, "Numeric summary для video basic features не вычислен.")

    if "tag" in set(video_basic_schema_df["column"]):
        tag_records = collect_lazy(scan_table(video_basic_path).select("tag").drop_nulls().limit(200_000)).to_dicts()
        tag_values = [str(row["tag"]) for row in tag_records]
        tag_ids = sorted({token for value in tag_values for token in re.findall(r"\\d+", value)})
        display(pd.DataFrame([{
            "rows_scanned": len(tag_values),
            "approx_unique_tag_ids_in_scanned_rows": len(tag_ids),
            "tag_examples": tag_values[:10],
            "tag_id_examples": tag_ids[:20],
            "source": "Вычислено по локальным файлам; approximate parsing тегов на ограниченной выборке",
        }]))


## Статистические признаки видео

Video statistic features (`show_cnt`, `play_cnt`, `play_duration`, `complete_play_cnt`, `valid_play_cnt`, `long_time_play_cnt`, `short_time_play_cnt`, `play_progress`, `like_cnt`, `comment_cnt`, `follow_cnt`, `share_cnt`, `collect_cnt`) - это агрегированные статистики видео. Это не individual interaction records.

In [ ]:
VIDEO_STAT_COLUMNS = (
    "show_cnt",
    "play_cnt",
    "play_duration",
    "complete_play_cnt",
    "valid_play_cnt",
    "long_time_play_cnt",
    "short_time_play_cnt",
    "play_progress",
    "like_cnt",
    "comment_cnt",
    "follow_cnt",
    "share_cnt",
    "collect_cnt",
)

video_stats_version = "Pure" if version_ready("Pure") else feature_version
video_stats_path = feature_table_path(video_stats_version, "video_statistics")
if video_stats_path is None:
    skip("таблица video statistic features недоступна")
else:
    shape, video_stats_schema_df = table_shape_and_schema(video_stats_path)
    display(pd.DataFrame([{**shape, "version": video_stats_version, "path": str(video_stats_path), "source": "Вычислено по локальным файлам"}]))
    display(video_stats_schema_df)

    video_stat_summary = numeric_column_summary_from_scan(video_stats_path, list(VIDEO_STAT_COLUMNS))
    table_or_message(video_stat_summary, "Numeric summary для video statistic features не вычислен.")

    if "show_cnt" in set(video_stats_schema_df["column"]):
        display(Markdown("### Топ видео по агрегированному show count"))
        display(pd.DataFrame(collect_lazy(
            scan_table(video_stats_path)
            .sort("show_cnt", descending=True)
            .select([col for col in ["video_id", "show_cnt", "play_cnt", "like_cnt", "comment_cnt"] if col in set(video_stats_schema_df["column"])])
            .limit(20)
        ).to_dicts()))


## Предупреждение о leakage

Video statistic features могут содержать информацию, агрегированную за период позже конкретного training interaction. Их нельзя автоматически подавать модели без проверки временного окна агрегации.

### Чеклист потенциального leakage

1. Future video statistics.
2. Popularity, рассчитанная с использованием future interactions.
3. Filtering до temporal split.
4. Смешивание random и standard logs без policy-aware design.
5. Статистики, построенные от target.
6. Preprocessing, fitted on full dataset до train-only fitting.
7. Temporal leakage через timestamps, upload dates или derived windows.
8. Item/user eligibility, определенная с использованием future information.

## Последовательная структура

Основной анализ последовательностей выполняется на KuaiRand-1K: standard logs объединяются, а взаимодействия каждого пользователя сортируются по `time_ms`.

In [ ]:
def sequence_analysis_1k() -> tuple[pd.DataFrame, pd.DataFrame]:
    lf = combine_interactions("1K", policies=("standard",))
    if lf is None:
        return pd.DataFrame(), pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if "user_id" not in names:
        return pd.DataFrame(), pd.DataFrame()

    exprs = [pl.len().alias("interactions")]
    if "time_ms" in names:
        time_col = pl.col("time_ms").cast(pl.Int64, strict=False)
        exprs.extend([time_col.min().alias("first_timestamp"), time_col.max().alias("last_timestamp")])
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("unique_videos"))

    per_user = pd.DataFrame(collect_lazy(lf.group_by("user_id").agg(exprs).sort("user_id")).to_dicts())
    if per_user.empty:
        return pd.DataFrame(), pd.DataFrame()

    summary = pd.DataFrame([{**numeric_summary(per_user["interactions"].to_numpy()), "source": "Вычислено по локальным файлам"}])
    sample_positions = np.linspace(0, len(per_user) - 1, num=min(10, len(per_user)), dtype=int)
    deterministic_users = per_user.iloc[sample_positions].copy()
    return summary, deterministic_users

sequence_summary_df, deterministic_users_df = sequence_analysis_1k()
table_or_message(sequence_summary_df, "Sequence-length summary для 1K не вычислен.")
table_or_message(deterministic_users_df, "Deterministic sample пользовательских sequences не вычислен.")


In [ ]:
if deterministic_users_df.empty:
    skip("для sequence-length plot нужны 1K standard logs")
else:
    lf_1k_standard = combine_interactions("1K", policies=("standard",))
    pl = require_polars()
    per_user_counts = pd.DataFrame(collect_lazy(
        lf_1k_standard.group_by("user_id").agg(pl.len().alias("interactions"))
    ).to_dicts())
    ax = per_user_counts["interactions"].hist(bins=60, figsize=(8, 4))
    ax.set_title("Распределение длины sequences в 1K standard")
    ax.set_xlabel("interactions per user")
    ax.set_ylabel("users")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()


## Sparsity и long tail

На KuaiRand-1K standard logs раздел считает interactions per user, unique videos per user, interactions per video и unique users per video. Approximate matrix sparsity считается по наблюдаемому user/item universe выбранных логов.

In [ ]:
def sparsity_long_tail_1k() -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    lf = combine_interactions("1K", policies=("standard",))
    if lf is None:
        return pd.DataFrame(), {}
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if not {"user_id", "video_id"}.issubset(names):
        return pd.DataFrame(), {}

    user_items = pd.DataFrame(collect_lazy(
        lf.group_by("user_id").agg([
            pl.len().alias("interactions_per_user"),
            pl.col("video_id").n_unique().alias("unique_videos_per_user"),
        ])
    ).to_dicts())
    item_users = pd.DataFrame(collect_lazy(
        lf.group_by("video_id").agg([
            pl.len().alias("interactions_per_video"),
            pl.col("user_id").n_unique().alias("unique_users_per_video"),
        ])
    ).to_dicts())
    observed_pairs = collect_lazy(lf.select(["user_id", "video_id"]).unique().select(pl.len().alias("pairs"))).to_dicts()[0]["pairs"]
    n_users = len(user_items)
    n_items = len(item_users)
    sparsity = 1 - observed_pairs / (n_users * n_items) if n_users and n_items else np.nan

    summary_rows = []
    for label, frame, column in [
        ("interactions per user", user_items, "interactions_per_user"),
        ("unique videos per user", user_items, "unique_videos_per_user"),
        ("interactions per video", item_users, "interactions_per_video"),
        ("unique users per video", item_users, "unique_users_per_video"),
    ]:
        summary_rows.append({"quantity": label, **numeric_summary(frame[column].to_numpy()), "source": "Вычислено по локальным файлам"})
    summary_rows.append({
        "quantity": "interaction matrix sparsity over observed 1K standard user/item universe",
        "count": observed_pairs,
        "min": None,
        "mean": sparsity,
        "median": None,
        "p75": None,
        "p90": None,
        "p95": None,
        "p99": None,
        "max": None,
        "source": "Вычислено по локальным файлам",
    })
    return pd.DataFrame(summary_rows), {"user_items": user_items, "item_users": item_users}

sparsity_summary_df, long_tail_tables = sparsity_long_tail_1k()
table_or_message(sparsity_summary_df, "Sparsity/long-tail summary не вычислен.")


In [ ]:
if not long_tail_tables:
    skip("для long-tail plots нужны 1K standard logs")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    long_tail_tables["user_items"]["interactions_per_user"].hist(ax=axes[0], bins=60)
    axes[0].set_title("Interactions per user")
    axes[0].set_yscale("log")

    long_tail_tables["item_users"]["interactions_per_video"].clip(
        upper=long_tail_tables["item_users"]["interactions_per_video"].quantile(0.99)
    ).hist(ax=axes[1], bins=60)
    axes[1].set_title("Interactions per video, p99 clipping только для графика")
    axes[1].set_yscale("log")
    plt.tight_layout()
    plt.show()


## Temporal EDA

Раздел аккуратно преобразует доступные date/timestamp fields и считает interactions by day для standard и random logs. Документированные окна сбора:

- 2022-04-08 to 2022-04-21
- 2022-04-22 to 2022-05-08

In [ ]:
def temporal_counts(version: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame(), pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    daily = pd.DataFrame()
    hourly = pd.DataFrame()

    if "date" in names:
        daily = pd.DataFrame(collect_lazy(
            lf.group_by(["policy", "date"]).agg(pl.len().alias("interactions")).sort(["date", "policy"])
        ).to_dicts())
        if not daily.empty:
            daily.insert(0, "version", version)
            daily["source"] = "Вычислено по локальным файлам"

    if "hourmin" in names:
        hourly_lf = lf.with_columns((pl.col("hourmin").cast(pl.Int64, strict=False) // 100).alias("hour"))
        hourly = pd.DataFrame(collect_lazy(
            hourly_lf.group_by(["policy", "hour"]).agg(pl.len().alias("interactions")).sort(["policy", "hour"])
        ).to_dicts())
        if not hourly.empty:
            hourly.insert(0, "version", version)
            hourly["source"] = "Вычислено по локальным файлам"

    return daily, hourly

temporal_version = "1K" if version_ready("1K") else "Pure"
daily_df, hourly_df = temporal_counts(temporal_version) if version_ready(temporal_version) else (pd.DataFrame(), pd.DataFrame())
table_or_message(daily_df, "Daily temporal counts не вычислены.")
table_or_message(hourly_df, "Hourly temporal counts не вычислены.")


In [ ]:
if daily_df.empty:
    skip("для daily timeline plot нужны computed daily counts")
else:
    plot_daily = daily_df.copy()
    plot_daily["date_plot"] = pd.to_datetime(plot_daily["date"].astype(str), errors="coerce")
    pivot = plot_daily.pivot_table(index="date_plot", columns="policy", values="interactions", aggfunc="sum").sort_index()
    ax = pivot.plot(figsize=(11, 4), marker="o")
    ax.axvspan(pd.Timestamp("2022-04-08"), pd.Timestamp("2022-04-21"), alpha=0.12, color="tab:blue")
    ax.axvspan(pd.Timestamp("2022-04-22"), pd.Timestamp("2022-05-08"), alpha=0.12, color="tab:orange")
    ax.set_title(f"Interactions by day: KuaiRand-{temporal_version}")
    ax.set_xlabel("date")
    ax.set_ylabel("interactions")
    plt.tight_layout()
    plt.show()

if hourly_df.empty:
    skip("для hourly plot нужны computed hourly counts")
else:
    pivot = hourly_df.pivot_table(index="hour", columns="policy", values="interactions", aggfunc="sum").sort_index()
    ax = pivot.plot(kind="bar", figsize=(11, 4), rot=0)
    ax.set_title(f"Interactions by hour: KuaiRand-{temporal_version}")
    ax.set_xlabel("hour")
    ax.set_ylabel("interactions")
    plt.tight_layout()
    plt.show()


## Полный KuaiRand-27K: результаты Slurm

Если `outputs/eda/27k_summary.json` существует, раздел загружает компактные результаты полного Slurm-запуска на cHARISMa. Это вычислено по локальным файлам / cHARISMa outputs, а не взято из документации. Raw 27K interaction logs здесь не загружаются.

Финальный полный запуск выполнен на V100: job `4253874`, partition `test`, constraint `type_a`, GRES `gpu:v100:1`, node `cn-012`, 10 CPU, `mem=0`, runtime `00:09:45`, MaxRSS `46537000K`.


In [ ]:
summary_27k = {}
tables_27k = {}


def read_27k_csv(stem: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f"27k_{stem}.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def show_27k_table(title: str, frame: pd.DataFrame, columns: list[str] | None = None, head: int | None = None) -> None:
    if frame.empty:
        print(f"Таблица 27K не найдена: {title}")
        return
    view = frame.copy()
    if columns is not None:
        existing = [column for column in columns if column in view.columns]
        view = view[existing]
    if head is not None:
        view = view.head(head)
    display(Markdown(f"### {title}"))
    display(view)


summary_27k_path = OUTPUT_DIR / "27k_summary.json"
if not summary_27k_path.exists():
    print(f"Вычисленный 27K summary не найден: {summary_27k_path}")
else:
    summary_27k = json.loads(summary_27k_path.read_text(encoding="utf-8"))
    table_stems = [
        "policy_counts",
        "source_counts",
        "standard_random_comparison",
        "feedback_summary",
        "binary_feedback_quality",
        "watch_time_summary",
        "duration_summary",
        "duration_by_tab",
        "duration_by_date",
        "play_ratio_summary",
        "sequence_summary",
        "max_user_sequence",
        "max_user_duplicate_summary",
        "item_popularity_summary",
        "item_universe_comparison",
        "feature_coverage",
        "duration_metadata_coverage",
        "duplicate_key_summary",
        "temporal_period_summary",
        "tab_summary",
        "daily",
        "hourly",
    ]
    tables_27k = {stem: read_27k_csv(stem) for stem in table_stems}

    display(pd.DataFrame([
        {
            "mode": summary_27k.get("mode"),
            "generated_at_utc": summary_27k.get("generated_at_utc"),
            "version_root": summary_27k.get("version_root"),
            "disk_size": summary_27k.get("disk_size"),
            "source": "Вычислено по локальным файлам через Slurm на cHARISMa",
        }
    ]))

    display(Markdown("### Параметры полного Slurm-запуска"))
    display(pd.DataFrame([
        {
            "job_id": "4253874",
            "state": "COMPLETED",
            "exit_code": "0:0",
            "partition": "test",
            "constraint": "type_a",
            "gres": "gpu:v100:1",
            "node": "cn-012",
            "cpus": 10,
            "memory": "mem=0",
            "runtime": "00:09:45",
            "max_rss": "46537000K",
        }
    ]))

    show_27k_table("Counts по policy", tables_27k["policy_counts"])
    show_27k_table("Counts по исходным файлам", tables_27k["source_counts"])
    show_27k_table("Сравнение standard и random", tables_27k["standard_random_comparison"])
    show_27k_table("Сводка feedback-сигналов", tables_27k["feedback_summary"])
    show_27k_table("Проверка binary feedback", tables_27k["binary_feedback_quality"])
    show_27k_table("Время просмотра", tables_27k["watch_time_summary"])
    show_27k_table("Проверка duration_ms <= 0", tables_27k["duration_summary"])
    show_27k_table("Сводка play_ratio", tables_27k["play_ratio_summary"])
    show_27k_table("Статистика последовательностей", tables_27k["sequence_summary"])
    show_27k_table("Самые длинные пользовательские истории", tables_27k["max_user_sequence"], head=10)
    show_27k_table("Дубли у пользователя с максимальной историей", tables_27k["max_user_duplicate_summary"])
    show_27k_table("Популярность видео", tables_27k["item_popularity_summary"])
    show_27k_table("Сопоставление item universe с README", tables_27k["item_universe_comparison"])
    show_27k_table("Покрытие user/video features", tables_27k["feature_coverage"])
    show_27k_table("Duration<=0 и наличие video metadata", tables_27k["duration_metadata_coverage"])
    show_27k_table("Проверка дублей по строгим ключам", tables_27k["duplicate_key_summary"])
    show_27k_table("Сводка по календарным периодам", tables_27k["temporal_period_summary"])
    show_27k_table("Tab/scenario summary", tables_27k["tab_summary"])


In [ ]:
if not summary_27k:
    skip("для графиков 27K нужен outputs/eda/27k_summary.json")
else:
    feedback = tables_27k.get("feedback_summary", pd.DataFrame())
    if feedback.empty:
        skip("для графика feedback нужны 27k_feedback_summary.csv")
    else:
        selected = ["is_click", "long_view", "is_like", "is_profile_enter", "is_hate"]
        plot_df = feedback[feedback["signal"].isin(selected)].pivot(index="signal", columns="policy", values="positive_rate")
        ax = (plot_df.loc[[signal for signal in selected if signal in plot_df.index]] * 100).plot(kind="barh", figsize=(9, 4))
        ax.set_title("KuaiRand-27K: доля положительных feedback-сигналов")
        ax.set_xlabel("доля, %")
        ax.set_ylabel("сигнал")
        plt.tight_layout()
        plt.show()

    play_ratio = tables_27k.get("play_ratio_summary", pd.DataFrame())
    share_cols = [
        "play_ratio_gt_1_share",
        "play_ratio_gt_2_share",
        "play_ratio_gt_5_share",
        "play_ratio_gt_10_share",
    ]
    if play_ratio.empty or not set(share_cols).issubset(play_ratio.columns):
        skip("для графика play_ratio нужны threshold share columns")
    else:
        plot_df = play_ratio.set_index("policy")[share_cols].rename(columns={
            "play_ratio_gt_1_share": "> 1",
            "play_ratio_gt_2_share": "> 2",
            "play_ratio_gt_5_share": "> 5",
            "play_ratio_gt_10_share": "> 10",
        }) * 100
        ax = plot_df.T.plot(kind="bar", figsize=(9, 4), rot=0)
        ax.set_title("KuaiRand-27K: доля строк с высоким play_ratio")
        ax.set_xlabel("порог play_ratio")
        ax.set_ylabel("доля, %")
        plt.tight_layout()
        plt.show()

    popularity = tables_27k.get("item_popularity_summary", pd.DataFrame())
    if popularity.empty:
        skip("для графика популярности нужны 27k_item_popularity_summary.csv")
    else:
        plot_df = popularity.set_index("policy")[["interactions_cv", "interactions_gini"]]
        ax = plot_df.plot(kind="bar", figsize=(8, 4), rot=0)
        ax.set_title("KuaiRand-27K: концентрация exposure по видео")
        ax.set_xlabel("policy")
        ax.set_ylabel("значение")
        plt.tight_layout()
        plt.show()

    daily = tables_27k.get("daily", pd.DataFrame())
    if daily.empty:
        skip("для daily plot нужен 27k_daily.csv")
    else:
        plot_df = daily.copy()
        plot_df["date_plot"] = pd.to_datetime(plot_df["date"].astype(str), format="%Y%m%d", errors="coerce")
        pivot = plot_df.pivot_table(index="date_plot", columns="policy", values="interactions", aggfunc="sum").sort_index()
        ax = pivot.plot(figsize=(10, 4), marker="o")
        ax.set_title("KuaiRand-27K: взаимодействия по дням")
        ax.set_xlabel("дата")
        ax.set_ylabel("взаимодействия")
        plt.tight_layout()
        plt.show()

    duration_by_tab = tables_27k.get("duration_by_tab", pd.DataFrame())
    if duration_by_tab.empty:
        skip("для tab duration plot нужен 27k_duration_by_tab.csv")
    else:
        plot_df = duration_by_tab.sort_values("interactions", ascending=False).head(12).copy()
        plot_df["policy_tab"] = plot_df["policy"].astype(str) + ": tab=" + plot_df["tab"].astype(str)
        ax = plot_df.set_index("policy_tab")["duration_non_positive_share"].mul(100).plot(kind="barh", figsize=(9, 5))
        ax.set_title("KuaiRand-27K: доля duration_ms <= 0 по крупнейшим tab")
        ax.set_xlabel("доля, %")
        ax.set_ylabel("policy/tab")
        plt.tight_layout()
        plt.show()


## Pure vs 1K vs 27K

Эта сравнительная таблица объединяет документированные характеристики датасета с размером файлов на диске, если он доступен локально. Полнота последовательностей, удобство разработки и пригодность для полного эксперимента — методологические заметки, а не вычисленные метрики.

In [ ]:
methodological_notes = pd.DataFrame([
    {
        "version": "KuaiRand-Pure",
        "Полнота последовательностей": "Фокус на candidate pool; полезно для анализа random/common items",
        "Удобство разработки": "Подходит для быстрой EDA по policy и debiasing",
        "Пригодность для полного эксперимента": "Полезно, когда задача требует Pure candidate pool",
    },
    {
        "version": "KuaiRand-1K",
        "Полнота последовательностей": "Плотные пользовательские истории для 1,000 пользователей",
        "Удобство разработки": "Лучшая версия для интерактивной разработки",
        "Пригодность для полного эксперимента": "Подходит для отладки протокола и проверок воспроизведения baseline",
    },
    {
        "version": "KuaiRand-27K",
        "Полнота последовательностей": "Полный крупномасштабный набор пользователей",
        "Удобство разработки": "Слишком тяжелый для полных интерактивных scans в ноутбуке",
        "Пригодность для полного эксперимента": "Основной кандидат для full-scale экспериментов",
    },
])
version_key = {"KuaiRand-Pure": "Pure", "KuaiRand-1K": "1K", "KuaiRand-27K": "27K"}

final_comparison = documentation_stats.rename(columns={
    "users": "Пользователи",
    "standard_items": "Items",
    "standard_interactions": "Standard-взаимодействия",
    "random_interactions": "Random-взаимодействия",
})[["version", "Пользователи", "Items", "Standard-взаимодействия", "Random-взаимодействия", "source"]]
final_comparison["version_key"] = final_comparison["version"].map(version_key)
disk_for_merge = disk_summary[["version", "disk_size", "source"]].rename(
    columns={"version": "version_key", "source": "Источник размера"}
)
final_comparison = final_comparison.merge(disk_for_merge, on="version_key", how="left")
final_comparison = final_comparison.merge(methodological_notes, on="version", how="left")
final_comparison = final_comparison.drop(columns=["version_key"])
final_comparison = final_comparison.rename(columns={
    "version": "Версия",
    "source": "Источник документации",
    "disk_size": "Размер на диске",
})
display(final_comparison)

# Протоколы train / validation / test

Сырой релиз KuaiRand сам по себе не задаёт один универсальный train/validation/test protocol для любой recommendation task.

Перед baseline нужно выбрать опубликованное benchmark family и точно воспроизвести preprocessing, split, candidate generation, negative sampling, filtering и evaluation.

In [ ]:
protocols = pd.DataFrame([
    {
        "Протокол": "Последовательный leave-one-out",
        "Train": "Для каждого пользователя: i1 ... i(n-2)",
        "Validation": "i(n-1)",
        "Test": "i(n)",
        "Подходящие задачи": "Next-item prediction, sequential recommendation",
        "Плюсы": "Сохраняет пользовательский временной порядок; часто используется в sequential recommenders",
        "Ограничения": "Может игнорировать глобальный календарный drift; чувствителен к filtering и коротким историям",
        "Условия сопоставимости": "Candidate set, negative sampling, минимальная длина истории и политика повторных items должны совпадать с published work",
    },
    {
        "Протокол": "Хронологический/date split",
        "Train": "Ранний календарный период",
        "Validation": "Более позднее validation-окно",
        "Test": "Финальное календарное окно",
        "Подходящие задачи": "Temporal generalization, production-like ranking",
        "Плюсы": "Сохраняет глобальное время и chronology deployment",
        "Ограничения": "Cold-start и изменяющийся item universe требуют явных правил",
        "Условия сопоставимости": "Границы дат, eligibility пользователей/items и feature time windows должны быть идентичны",
    },
    {
        "Протокол": "Standard training + random-exposure evaluation",
        "Train": "Взаимодействия standard-policy",
        "Validation": "Зависит от задачи: standard или held-out random exposure",
        "Test": "Random-exposure компонент, если он уместен для задачи",
        "Подходящие задачи": "Debiased recommendation, off-policy evaluation, causal/random-exposure evaluation",
        "Плюсы": "Использует randomized exposure для снижения policy-selection bias в evaluation",
        "Ограничения": "Random log не становится автоматически правильным test set для любой цели",
        "Условия сопоставимости": "Exposure policy, candidate pool, IPS/causal estimators и определение target должны совпадать",
    },
])
display(protocols)

## Возможные задачи


In [ ]:
tasks = pd.DataFrame([
    {"Задача": "предсказание следующего item", "Целевая переменная": "следующий video_id", "Входные данные": "упорядоченная история пользователя", "Рекомендуемая версия KuaiRand": "1K для разработки, 27K для полного эксперимента", "Возможные метрики": "Recall@K, NDCG@K, HitRate@K, MRR"},
    {"Задача": "top-K рекомендация", "Целевая переменная": "отложенный positive item", "Входные данные": "история пользователя, item universe, candidate set", "Рекомендуемая версия KuaiRand": "сначала 1K, затем 27K", "Возможные метрики": "Recall@K, NDCG@K, HitRate@K"},
    {"Задача": "предсказание клика", "Целевая переменная": "is_click", "Входные данные": "пользователь, item, контекст, история", "Рекомендуемая версия KuaiRand": "Pure/1K для EDA, 27K для масштаба", "Возможные метрики": "AUC, LogLoss"},
    {"Задача": "предсказание long_view", "Целевая переменная": "long_view", "Входные данные": "watch-time контекст и item metadata", "Рекомендуемая версия KuaiRand": "Pure/1K", "Возможные метрики": "AUC, LogLoss, calibration"},
    {"Задача": "multi-task предсказание feedback", "Целевая переменная": "click, like, follow, comment, forward, hate, long_view", "Входные данные": "общая user/item/context репрезентация", "Рекомендуемая версия KuaiRand": "сначала 1K, затем 27K", "Возможные метрики": "AUC/LogLoss по каждому target, ranking metrics для derived preference"},
    {"Задача": "debiased recommendation", "Целевая переменная": "policy-aware positive feedback", "Входные данные": "standard logs плюс random exposure", "Рекомендуемая версия KuaiRand": "Pure и 27K", "Возможные метрики": "IPS/SNIPS variants, NDCG@K under defined candidate policy"},
    {"Задача": "off-policy evaluation", "Целевая переменная": "counterfactual policy value", "Входные данные": "randomized exposure и logged feedback", "Рекомендуемая версия KuaiRand": "Pure/27K", "Возможные метрики": "IPS, SNIPS, DR, если propensities/design это позволяют"},
    {"Задача": "evaluation на random exposure", "Целевая переменная": "feedback при random candidate exposure", "Входные данные": "random log", "Рекомендуемая версия KuaiRand": "Pure/27K", "Возможные метрики": "AUC, Recall@K/NDCG@K с явно заданным candidate set"},
    {"Задача": "causal recommendation", "Целевая переменная": "causal estimand, заданный до моделирования", "Входные данные": "random exposure, standard exposure, covariates", "Рекомендуемая версия KuaiRand": "Pure/27K", "Возможные метрики": "метрики error/value под заданный estimand"},
])
display(tasks)

## Метрики

Потенциальные метрики:

- `Recall@K`
- `NDCG@K`
- `HitRate@K`
- `MRR`
- `AUC` / `LogLoss`, если цель — предсказание вероятностей

Одинаковое название метрики не означает сопоставимый benchmark. Например, `NDCG@10` зависит от split, candidate set, negative sampling, filtering, evaluation users, preprocessing, repeated-item policy и того, участвует ли random exposure в evaluation. Это критично для будущей SOTA-таблицы.

# Что мы узнали о KuaiRand

Ячейка ниже формирует выводы по вычисленным таблицам, если они доступны. Если ноутбук запущен без файлов KuaiRand, она сообщает, что вычисленные выводы ожидают запуска на данных, а не придумывает числа.

In [ ]:
conclusions = [
    "Статистика из документации / README показывает, что 1K - практичная версия для разработки, 27K - кандидат для полного эксперимента, а Pure фокусируется на common/random candidate-pool setting.",
    "KuaiRand содержит standard-policy logs и randomized exposure logs; их нужно моделировать как разные политики сбора данных, а не автоматически переименовывать в train/test.",
    "Random exposure важен для изучения exposure bias, selection bias, debiasing, off-policy evaluation и causal recommendation, но только при явно заданном дизайне задачи.",
    "Ноутбук рассматривает video statistic features как потенциально leaky агрегированные признаки до проверки их временного окна.",
    "Выбор benchmark protocol критичен: preprocessing, split, candidates, negative sampling и evaluation users должны совпадать с выбранным семейством опубликованных baselines.",
]

if not policy_counts_df.empty:
    for version in sorted(policy_counts_df["Version"].unique()):
        sub = policy_counts_df[policy_counts_df["Version"] == version]
        policies = ", ".join(sorted(sub["Policy"].astype(str).tolist()))
        conclusions.append(f"Вычислено по локальным файлам: KuaiRand-{version} содержит policy-сводки для: {policies}.")
else:
    conclusions.append("Вычисленные policy counts ожидают запуска на данных: файлы Pure/1K недоступны в DATA_ROOT.")

if summary_27k:
    policy_27k = tables_27k.get("policy_counts", pd.DataFrame())
    if not policy_27k.empty:
        standard_row = policy_27k[policy_27k["policy"] == "standard"].iloc[0]
        random_row = policy_27k[policy_27k["policy"] == "random"].iloc[0]
        conclusions.append(
            "Полный 27K EDA вычислен на cHARISMa: "
            f"standard={int(standard_row['interactions']):,} взаимодействий, "
            f"random={int(random_row['interactions']):,} взаимодействий."
        )
    universe_27k = tables_27k.get("item_universe_comparison", pd.DataFrame())
    if not universe_27k.empty:
        values = dict(zip(universe_27k["metric"], universe_27k["count"]))
        conclusions.append(
            "Расхождение 32 items подтверждено: "
            f"README/feature universe={int(values['README standard items']):,}, "
            f"observed standard video_id={int(values['Unique standard interaction video_id']):,}; "
            "все observed items покрыты feature-файлами."
        )
    duration_27k = tables_27k.get("duration_summary", pd.DataFrame())
    if not duration_27k.empty:
        standard_duration = duration_27k[duration_27k["policy"] == "standard"].iloc[0]
        random_duration = duration_27k[duration_27k["policy"] == "random"].iloc[0]
        conclusions.append(
            "duration_ms <= 0 состоит из нулей: "
            f"standard={int(standard_duration['duration_non_positive_count']):,} "
            f"({standard_duration['duration_non_positive_share']:.3%}), "
            f"random={int(random_duration['duration_non_positive_count']):,} "
            f"({random_duration['duration_non_positive_share']:.3%})."
        )
    play_27k = tables_27k.get("play_ratio_summary", pd.DataFrame())
    if not play_27k.empty:
        standard_play = play_27k[play_27k["policy"] == "standard"].iloc[0]
        random_play = play_27k[play_27k["policy"] == "random"].iloc[0]
        conclusions.append(
            "play_ratio > 1 массово встречается и не должен удаляться как ошибка без отдельного правила: "
            f"standard={standard_play['play_ratio_gt_1_share']:.3%}, random={random_play['play_ratio_gt_1_share']:.3%}."
        )
    popularity_27k = tables_27k.get("item_popularity_summary", pd.DataFrame())
    if not popularity_27k.empty:
        standard_pop = popularity_27k[popularity_27k["policy"] == "standard"].iloc[0]
        random_pop = popularity_27k[popularity_27k["policy"] == "random"].iloc[0]
        conclusions.append(
            "Random exposure намного ровнее standard: "
            f"Gini random={random_pop['interactions_gini']:.3f}, "
            f"Gini standard={standard_pop['interactions_gini']:.3f}."
        )
else:
    conclusions.append("Полные выводы по KuaiRand-27K ожидают `outputs/eda/27k_summary.json` и компактные 27k CSV.")

if not sequence_summary_df.empty:
    row = sequence_summary_df.iloc[0]
    conclusions.append(
        "Вычислено по локальным файлам: доступна сводка длины 1K standard user histories "
        f"с median={row.get('median')} и p95={row.get('p95')} взаимодействий."
    )
else:
    conclusions.append("Вычисленные выводы по длинам последовательностей ожидают доступности KuaiRand-1K.")

if not sparsity_summary_df.empty:
    sparsity_row = sparsity_summary_df[sparsity_summary_df["quantity"].str.contains("sparsity", na=False)]
    if not sparsity_row.empty:
        conclusions.append(
            "Вычислено по локальным файлам: approximate interaction-matrix sparsity для наблюдаемого 1K standard universe равна "
            f"{sparsity_row.iloc[0]['mean']}."
        )
else:
    conclusions.append("Вычисленные выводы по sparsity и long tail ожидают доступности standard-логов KuaiRand-1K.")

if not feedback_rates_df.empty:
    available_signals = ", ".join(sorted(feedback_rates_df["signal"].unique()))
    conclusions.append(f"Вычислено по локальным файлам: feedback-сводка доступна для {available_signals}.")
else:
    conclusions.append("Вычисленные выводы по feedback rates ожидают доступности логов взаимодействий.")

if not shift_summary_df.empty:
    conclusions.append("Вычислено по локальным файлам: доступна сводка distribution difference между standard и random для проверенных версий.")
else:
    conclusions.append("Выводы по distribution difference между standard и random ожидают доступности логов Pure/1K.")

conclusions.extend([
    "Pure полезен для исследований candidate pool и random-exposure-focused задач.",
    "1K полезен для интерактивной разработки, отладки protocol и проверок воспроизведения baseline.",
    "27K нужно использовать для full-scale экспериментов через batch/lazy scripts, а не через полные интерактивные scans в ноутбуке.",
    "Перед baseline-работой нужно выбрать конкретные опубликованные статьи и точно воспроизвести их preprocessing, split, candidate construction и evaluation.",
])

conclusions = conclusions[:15]
display(Markdown("\n".join(f"{idx + 1}. {text}" for idx, text in enumerate(conclusions))))


# Следующие шаги

1. Выбрать опубликованное benchmark-семейство.
2. Выбрать конкретную strong baseline / SOTA-статью.
3. Воспроизвести preprocessing.
4. Воспроизвести split.
5. Воспроизвести evaluation protocol.
6. Получить baseline metrics.
7. Сравнить с опубликованными результатами.
8. Только после этого проектировать новый method.
9. Провести ablations.
10. Перейти к full 27K experiments.

## Как запускать 27K

Полный 27K scan нужно запускать через `src/eda_27k.py`, желательно через `slurm/eda_27k.sh`. Ноутбук не использует `pandas.read_csv` для 27K и не материализует полные 27K interaction logs.